# ⚙️ Automate Your Synthetic Data Pipeline

In this lesson, we will cover the importance of writing code for synthetic data generation that is reusable and with configurable components that can be rerun reproducibly.

In Video 3 we built a synthesis pipeline for the Bank Marketing dataset — load, prepare, train, generate, validate. The code worked, but it was written as a linear sequence of cells with hard-coded paths, sizes, and parameters scattered throughout.

Here we take that same pipeline and reorganise it into a form that is ready to automate:

1. A `PipelineConfig` dataclass — all parameters in one place
2. Four pipeline functions — one responsibility each
3. An orchestrator — one call to run everything
4. A JSON run log — timestamp, config snapshot, and metrics per run

## 1. Pipeline Configuration

Everything that was hard-coded in Video 3 — the dataset ID, which columns to drop, how many rows to generate, where to save the output — lives here now. Changing a run means changing the config, not hunting through code.

Using a `dataclass` rather than a plain dictionary gives us type hints, a clean repr for logging, and default values that make the config self-documenting.

In [5]:
from dataclasses import dataclass, field

@dataclass
class PipelineConfig:
    # Data source
    dataset_id: int   = 222                                    # UCI dataset ID
    dataset_path: str = 'data/bank-full.csv'
    drop_columns: list = field(default_factory=lambda: ['duration'])  # leakage columns

    # Anonymization — None for public datasets; dict of column → faker method for real pipelines
    anonymize: dict   = None

    # Generation
    n_samples: int    = 45211                                  # rows to generate
    seed: int         = 42

    # Constraints
    min_age: int      = 18
    min_campaign: int = 1

    # Output
    output_dir: str   = 'output/runs'
    run_name: str     = 'bank_marketing'


config = PipelineConfig()
print(config)

PipelineConfig(dataset_id=222, dataset_path='data/bank-full.csv', drop_columns=['duration'], anonymize=None, n_samples=45211, seed=42, min_age=18, min_campaign=1, output_dir='output/runs', run_name='bank_marketing')


## 2. Pipeline Functions

The four steps from Video 3 each become a function with a single responsibility. The `config` object is the only thing shared between them — no globals, no hidden state.

In [14]:
from ydata.dataset import Dataset
from ydata.metadata import Metadata

def prepare_data(config: PipelineConfig):
    """Load the dataset, drop leakage columns, and define calculated features."""
    import pandas as pd

    df = pd.read_csv(config.dataset_path, sep=';')
    df = df.drop(columns=config.drop_columns, errors='ignore')

    calculated_features = [
        {
            'calculated_features': 'total_contacts',
            'function': lambda campaign, previous: (campaign + previous).values,
            'calculated_from': ['campaign', 'previous'],
        }
    ]

    dataset  = Dataset(df)
    metadata = Metadata(dataset)

    print(f'prepare_data ✓  {df.shape[0]:,} rows × {df.shape[1]} columns')
    return dataset, metadata, calculated_features

In [15]:
from ydata.synthesizers.regular import RegularSynthesizer

def train(dataset, metadata, calculated_features, config: PipelineConfig):
    """Fit the synthesizer on the prepared dataset."""
    synth = RegularSynthesizer()
    synth.fit(
        X=dataset,
        metadata=metadata,
        anonymize=config.anonymize,
        calculated_features=calculated_features,
    )
    print('train ✓')
    return synth

In [16]:
def generate(synth, config: PipelineConfig):
    """Sample synthetic records from the trained synthesizer."""
    synth_df = synth.sample(n_samples=config.n_samples).to_pandas()
    print(f'generate ✓  {len(synth_df):,} rows generated')
    return synth_df

In [21]:
from ydata.constraints import ConstraintEngine, GreaterThan, CustomConstraint

def validate(synth_df, config: PipelineConfig):
    """Apply business constraints and return the clean dataset and a metrics dict."""
    engine = ConstraintEngine([
        GreaterThan(
            columns=['campaign'],
            value=config.min_campaign - 1,
        ),
        GreaterThan(
            columns=['age'],
            value=config.min_age - 1,
        ),
        CustomConstraint(
            lambda row: not (row['pdays'] == -1 and row['poutcome'] != 'unknown'),
            columns=['pdays', 'poutcome'],
            name='no_contact_no_outcome',
            axis='row',
        ),
    ])

    synth_dataset = Dataset(synth_df)
    engine.validate(synth_dataset)
    clean_df = engine.filter(synth_dataset).to_pandas()

    metrics = {
        'rows_before_filter': len(synth_df),
        'rows_after_filter':  len(clean_df),
        'rows_removed':       len(synth_df) - len(clean_df),
        'pass_rate':          round(len(clean_df) / len(synth_df), 4),
    }

    print(f'validate ✓  {metrics["rows_after_filter"]:,} rows passed '
          f'({metrics["pass_rate"] * 100:.1f}%)')
    return clean_df, metrics

## 3. Run Logger

Each run writes a JSON file containing the timestamp, the full config, and the validation metrics. Two runs with different parameters produce two log files — you can compare them directly to understand what changed and why.

In [22]:
def log_run(config: PipelineConfig, metrics: dict, output_path: str) -> str:
    """Write a JSON log file for this run and return its path."""
    log = {
        'timestamp': datetime.now().isoformat(),
        'config':    asdict(config),
        'metrics':   metrics,
        'output':    output_path,
    }
    timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_path   = f'{config.output_dir}/{config.run_name}_{timestamp}.json'
    os.makedirs(config.output_dir, exist_ok=True)

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2, default=str)

    return log_path

## 4. Orchestrator

`run_pipeline()` calls the four functions in sequence and handles saving and logging. The entire pipeline is now one function call.

In [31]:
import os 
from datetime import datetime
from dataclasses import asdict
import json

def run_pipeline(config: PipelineConfig) -> dict:
    """Execute the full synthetic data pipeline and return the run log."""
    print(f'\n▶  Pipeline: {config.run_name}')
    print(f'   n_samples={config.n_samples:,}  seed={config.seed}  output={config.output_dir}\n')

    dataset, metadata, calculated_features = prepare_data(config)
    synth    = train(dataset, metadata, calculated_features, config)
    synth_df = generate(synth, config)
    clean_df, metrics = validate(synth_df, config)

    # Save synthetic dataset
    os.makedirs(config.output_dir, exist_ok=True)
    timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = f'{config.output_dir}/{config.run_name}_{timestamp}.csv'
    clean_df.to_csv(output_path, index=False)

    # Log the run
    log_path = log_run(config, metrics, output_path)

    print(f'\n✓  Dataset → {output_path}')
    print(f'✓  Log     → {log_path}')
    return {'metrics': metrics, 'output': output_path, 'log': log_path}

## 5. Run the Pipeline

In [32]:
# Default run — same parameters as Video 3
config = PipelineConfig()
run_1  = run_pipeline(config)


▶  Pipeline: bank_marketing
   n_samples=45,211  seed=42  output=output/runs



Done:  80%|████████████████████████████▊       | 4/5 [00:00<00:00,  4.21step/s]

prepare_data ✓  45,211 rows × 16 columns
Validate the segmentation strategy.
INFO: 2026-08-11 15:17:41,073 [SYNTHESIZER] - Number columns considered for synth: 16


INFO: 2026-08-11 15:17:41,444 [SYNTHESIZER] - Starting the synthetic data modeling process over 1x1 blocks.
INFO: 2026-08-11 15:17:41,447 [SYNTHESIZER] - Preprocess segment
INFO: 2026-08-11 15:17:41,450 [SYNTHESIZER] - Synthesizer init.
INFO: 2026-08-11 15:17:41,450 [SYNTHESIZER] - Processing the data prior fitting the synthesizer.
train ✓
INFO: 2026-08-11 15:17:44,776 [SYNTHESIZER] - Start generating model samples.
INFO: 2026-08-11 15:17:46,021 [SYNTHESIZER] - Postprocessing.
INFO: 2026-08-11 15:17:46,025 [SYNTHESIZER] - Numerical clipping
INFO: 2026-08-11 15:17:46,055 let write into the connector None
INFO: 2026-08-11 15:17:46,055 using kwargs {}
generate ✓  45,211 rows generated
validate ✓  45,211 rows passed (100.0%)

✓  Dataset → output/runs/bank_marketing_20260811_151746.csv
✓  Log     → output/runs/bank_marketing_20260811_151746.json


## 6. Compare Runs

Because every parameter lives in `PipelineConfig`, running a different experiment means creating a new config — not editing cells. The log files make the difference between runs traceable.

In [33]:
# Second run — half the sample size, different seed
config_2 = PipelineConfig(
    n_samples=20000,
    seed=99,
    run_name='bank_marketing_small',
)
run_2 = run_pipeline(config_2)


▶  Pipeline: bank_marketing_small
   n_samples=20,000  seed=99  output=output/runs



Done:  80%|████████████████████████████▊       | 4/5 [00:00<00:00,  4.65step/s]

prepare_data ✓  45,211 rows × 16 columns
Validate the segmentation strategy.
INFO: 2026-08-11 15:18:04,859 [SYNTHESIZER] - Number columns considered for synth: 16


INFO: 2026-08-11 15:18:05,221 [SYNTHESIZER] - Starting the synthetic data modeling process over 1x1 blocks.
INFO: 2026-08-11 15:18:05,223 [SYNTHESIZER] - Preprocess segment
INFO: 2026-08-11 15:18:05,225 [SYNTHESIZER] - Synthesizer init.
INFO: 2026-08-11 15:18:05,226 [SYNTHESIZER] - Processing the data prior fitting the synthesizer.
train ✓
INFO: 2026-08-11 15:18:08,557 [SYNTHESIZER] - Start generating model samples.
INFO: 2026-08-11 15:18:09,600 [SYNTHESIZER] - Postprocessing.
INFO: 2026-08-11 15:18:09,603 [SYNTHESIZER] - Numerical clipping
INFO: 2026-08-11 15:18:09,622 let write into the connector None
INFO: 2026-08-11 15:18:09,623 using kwargs {}
generate ✓  20,000 rows generated
validate ✓  20,000 rows passed (100.0%)

✓  Dataset → output/runs/bank_marketing_small_20260811_151809.csv
✓  Log     → output/runs/bank_marketing_small_20260811_151809.json


In [35]:
import pandas as pd

# Compare the two runs side by side
comparison = pd.DataFrame([
    {'run': 'run_1', **run_1['metrics']},
    {'run': 'run_2', **run_2['metrics']},
])
print(comparison.to_string(index=False))

  run  rows_before_filter  rows_after_filter  rows_removed  pass_rate
run_1               45211              45211             0        1.0
run_2               20000              20000             0        1.0


> **⏸ Active Learning** — Look back at your notebook from Video 3. How many hard-coded values can you count? Dataset path, number of samples, output filename, constraint thresholds — each one is a parameter that would need to be manually edited for every new experiment. Now look at `PipelineConfig` above: how would you adapt it to your own synthetic data use case?

## ✅ Summary

In this notebook you:

1. **Defined a `PipelineConfig` dataclass** — a single source of truth for every parameter in the pipeline
2. **Refactored a synthetic data pipeline into four functions** — `prepare_data`, `train`, `generate`, `validate` — each with one responsibility
3. **Built a `run_pipeline()` orchestrator** — the entire pipeline is now a single, reproducible function call
4. **Created a run log** — every run records its timestamp, config, and output metrics, making experiments comparable

This structure is the foundation for automation. The next step — if your workflow demands it — is to schedule `run_pipeline()` with an orchestration tool like Argo Workflows or Kubeflow Pipelines, which add scheduling, versioning, and integration with broader ML pipelines on top of exactly this kind of modular, config-driven code.